<a href="https://colab.research.google.com/github/OAKVISUALZ/MLC_PROJECT/blob/main/FineTuned_LLM_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install dependencies

In [3]:
!pip install transformers datasets huggingface_hub

### Data Loading and Preprocessing

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load the dataset using the fully qualified repository path
dataset = load_dataset("rajpurkar/squad")

def add_end_of_text(example):
    example['question'] = example['question'] + '<|endoftext|>'
    return example

# Clean and map the text
dataset = dataset.remove_columns(['id', 'title', 'context', 'answers'])
dataset = dataset.map(add_end_of_text)

# Initialize Tokenizer
model_checkpoint = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

def tokenize_function(examples):
    return tokenizer(examples["question"], truncation=True)

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=["question"]
)

# Group texts into blocks
block_size = 128

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)

# Create smaller subsets for a quick training run
small_train_dataset = lm_datasets["train"].shuffle(seed=42).select(range(100))
small_eval_dataset = lm_datasets["validation"].shuffle(seed=42).select(range(100))

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map (num_proc=4):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10570 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/87599 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/10570 [00:00<?, ? examples/s]

### Model Training

In [5]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
import math

# Define the model checkpoint locally to prevent NameError
model_checkpoint = "distilgpt2"

# Load the base model
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

training_args = TrainingArguments(
    output_dir=f"{model_checkpoint}-squad",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False, # Set to True if you log in to Hugging Face
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
)

# Start training
trainer.train()

# Evaluate and calculate Perplexity
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.695931
2,No log,3.520411
3,No log,3.462316


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
No log,3.462316,3


Perplexity: 31.89


### Text Generation

In [6]:
start_text = (
    "A speedrun is a playthrough of a video game, "
    "or section of a video game, with the goal of "
    "completing it as fast as possible. Speedruns "
    "often follow planned routes, which may incorporate sequence "
    "breaking, and might exploit glitches that allow sections to "
    "be skipped or completed more quickly than intended. "
)
prompt = "What is the"

# Ensure inputs are moved to the same device as the model
inputs = tokenizer(
     start_text + prompt,
     add_special_tokens=False,
     return_tensors="pt"
).to(model.device)

outputs = model.generate(
     inputs["input_ids"],
     max_length=100,
     do_sample=True,
     top_k=50,
     top_p=0.95,
     temperature=0.9,
     num_return_sequences=3
)

print("--- Generated Outputs ---")
for i, output in enumerate(outputs):
    generated_text = tokenizer.decode(output, skip_special_tokens=True)
    print(f"\nSequence {i+1}:\n{generated_text}")

--- Generated Outputs ---

Sequence 1:
A speedrun is a playthrough of a video game, or section of a video game, with the goal of completing it as fast as possible. Speedruns often follow planned routes, which may incorporate sequence breaking, and might exploit glitches that allow sections to be skipped or completed more quickly than intended. What is the purpose of Speedruns?

Sequence 2:
A speedrun is a playthrough of a video game, or section of a video game, with the goal of completing it as fast as possible. Speedruns often follow planned routes, which may incorporate sequence breaking, and might exploit glitches that allow sections to be skipped or completed more quickly than intended. What is the speedrun?

Sequence 3:
A speedrun is a playthrough of a video game, or section of a video game, with the goal of completing it as fast as possible. Speedruns often follow planned routes, which may incorporate sequence breaking, and might exploit glitches that allow sections to be skipped